In [ ]:
import numpy as np

from data import prepare_data_paths, read_preprocessed_data, save_preprocessed_data

# Apply feature selection and save the preprocessed array

In [ ]:
target_prefix = "merged_support3"
n_preliminary = 1000000
n_secondary = 8192

preliminary_selection_file = "../cache/43034818_seed42_cv42_outerfold0_variance_feature_variance.npy"
secondary_selection_file = "../cache/43034818_seed42_cv42_outerfold0_variance_1000000_1000000_xgb_basic_feature_importance_mean.npy"

output_prefix = f"{target_prefix}_variance_{n_preliminary}_xgb_{n_secondary}"

In [ ]:
preprocess_path, label_file = prepare_data_paths()
X, variant_info_df = read_preprocessed_data(preprocess_path / target_prefix)

In [ ]:
variances = np.load(preliminary_selection_file)

preliminary_selected_indices = np.argsort(variances)[-n_preliminary:]

boolean_mask = np.zeros(X.shape[1], dtype=bool)
boolean_mask[preliminary_selected_indices] = True
X_selected_preliminary = X[:, boolean_mask]
variant_info_df = variant_info_df.iloc[boolean_mask]

assert boolean_mask.sum() == X_selected_preliminary.shape[1]
print(f"Selected variants based on variance filter. Shape: {X_selected_preliminary.shape}")

In [ ]:
secondary_feature_importance = np.load(secondary_selection_file)
secondary_selected_indices = np.argsort(secondary_feature_importance)[-n_secondary:][::-1]
X_selected_secondary = X_selected_preliminary[:, secondary_selected_indices]
variant_info_df = variant_info_df.iloc[secondary_selected_indices].reset_index(drop=True)

assert secondary_feature_importance.shape[0] == X_selected_preliminary.shape[1]
print(f"Selected variants based on XGB. Shape: {X_selected_secondary.shape}")

In [ ]:
save_preprocessed_data(X_selected_secondary, variant_info_df, preprocess_path / output_prefix)